# EDA of US Soybean

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import date
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import linregress
import numpy as np
from statsmodels.tsa.stattools import adfuller


In [ ]:

yield_df = pd.read_csv("../data/yield.csv")

print(yield_df.info())
print(yield_df.describe())
yield_df.value_counts("state_name").plot(kind="bar")


In [ ]:

print(yield_df["state_name"].nunique())


In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(data=yield_df, x='state_name', y='yield')
plt.xticks(rotation=90)
plt.title('Yield by State')
plt.show()

Data is balanced, except for Florida & West Virginia which have only 14 years of data. 
'Other states' have only 4 occurences (2015 to 2018) and yield is close to 0.

I will drop the 'Other states' category.

In [ ]:
yield_df = yield_df[yield_df["state_name"] != "OTHER STATES"].reset_index(drop=True)

In [ ]:

tp_df = pd.read_csv("../data/tp.csv")
tmax_df = pd.read_csv("../data/2t.csv")

features = tp_df.merge(tmax_df, on=["state_name", "date"])

print(features.info())
print(features.describe())

In [ ]:
print(features["state_name"].nunique())
print(features["state_name"].unique().tolist())


Missing North Dakota & Arkansas that are top 10 producing states.

In [ ]:

features["date"] = pd.to_datetime(features["date"])
print(features.describe())


Yield data goes from 2005 to 2022 while features go from 2005 to 2023, yet 2023 is incomplete (2023-09-19), either drop it or use it as validation set if soybean harvest date ends before mid-September in US.

There are no missing values to impute.

In [ ]:
start_date = date(2005, 1, 1)
end_date = date(2023, 9, 19)
delta = end_date - start_date
number_of_days = delta.days + 1  # +1 to include both start and end dates

print(len(features["date"].unique()))

print(number_of_days)

min_date = features["date"].min()
max_date = features["date"].max()
complete_range = pd.date_range(start=min_date, end=max_date, freq="D")

missing_dates = complete_range.difference(features["date"])

print("Missing dates:")
print(missing_dates)


9 days are missing from the 6827 daily weather data points. We could impute missing data by calling weather API.

In [ ]:
plt.figure(figsize=(10, 6))
sns.lineplot(data=yield_df, x='year', y='yield', marker='o')


slope, intercept, r_value, p_value, std_err = linregress(yield_df['year'], yield_df['yield'])
sns.regplot(data=yield_df, x='year', y='yield', scatter=False, color='black', ci=None, label=f'Global Trend (r={r_value:.2f})')
plt.title('US Soybean Yield per Year')
plt.xlabel('Year')
plt.ylabel('Yield (bu/A)')
plt.tight_layout()
plt.show()

In [ ]:

plt.figure(figsize=(8, 6))

# Plot all state points
sns.scatterplot(data=yield_df, x='year', y='yield', alpha=0.6, color='lightblue', legend=False)

# Plot aggregated mean yield per year
yield_aggregated = yield_df.groupby('year')['yield'].mean().reset_index()
yield_aggregated = yield_aggregated.sort_values('year')
sns.scatterplot(data=yield_aggregated, x='year', y='yield', color='orange', label='Annual Mean Yield')

# Fit a global linear regression line
slope, intercept, r_value, p_value, std_err = linregress(yield_df['year'], yield_df['yield'])
sns.regplot(data=yield_df, x='year', y='yield', scatter=False, color='black', ci=None, label=f'Global Trend (r={r_value:.2f})')

plt.title('Yield Aggregated by State Over Time with Global Trend')
plt.xlabel('Year')
plt.ylabel('Yield (bu/A)')

# Place legend below the plot
plt.legend(title='Legend', loc='upper left', ncol=3)

# Ensure x-axis labels are integers for years
plt.xticks(sorted(yield_df['year'].unique()), rotation=90)

plt.grid(True)
plt.tight_layout()
plt.show()


yield_aggregated = yield_df.groupby('year')['yield'].mean().reset_index()
adf_result = adfuller(yield_aggregated['yield'].dropna())
print('ADF Statistic:', adf_result[0])
print('p-value:', adf_result[1])


I will however include year as a proxy for technological advances in genetics and field management.

In [ ]:
x = yield_aggregated['year']
y = yield_aggregated['yield']
model = np.polyfit(x, y, 1)
predicted = np.polyval(model, x)
y_detrended = y - predicted

sns.scatterplot(x=x, y=y_detrended, color='green', label='Detrended Yield')
plt.axhline(y=0, color='black', linestyle='--')
plt.title(f'(b) Average yield (2003-2019) = {y.mean():.2f} t/ha')
plt.xlabel('Year')
plt.ylabel('Detrended Yield (t/ha)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.scatter(soybean_yield["yield"], total_precipitation["pre"], s=sizes, c=colors, vmin=0, vmax=100)
